### Get the risk and sentiment text

##### Download the top100_comp_sec_filings table

In [0]:
# create a spark session
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("trading_sentiment_platform").getOrCreate()

# download the top100 companies table and display
top100_raw_files = spark.read.table("workspace.sec_filings.top100_comp_sec_filings")
display(top100_raw_files.limit(5))

##### extract the risk and management discussion texts

In [0]:
# define a function that scrapes through the file's URLs and extracts:
# (1) entire filing text
# (2) Risk Factors text (Item 1A)
# (3) MD&A text (Item 7 or Item 2 for small filers)
def parsing(filing_url):
    from bs4 import BeautifulSoup
    import requests
    import re

    # create a User-Agent header to access SEC filings
    headers = {"User-Agent": "Joel Doh joeljuniordoh19@gmail.com"}

    try:
        # Request the filing page
        response = requests.get(filing_url, headers=headers)

        # If HTTP request fails, return empty fields
        if response.status_code != 200:
            return ("", "", "")
        
        # extract all text from HTML, collapse tags with spaces
        soup = BeautifulSoup(response.text, "html.parser")
        text = soup.get_text(separator=" ", strip=True)

        # remove extra whitespace/newlines
        # convert to uppercase to standardize for regex matching
        cleaned_text = re.sub(r"\s+", " ", text)
        upper_text = cleaned_text.upper()

        # ------------------------------
        # Extract RISK FACTORS section
        # ------------------------------

        upper_text_risk = None

        # Find all occurrences of “ITEM 1A RISK FACTORS”
        risk_iter = list(re.finditer(r"ITEM\s*1A\.?\s*RISK FACTORS", upper_text))

        # If found more than once, start from the 2nd occurrence  
        # (the first is sometimes in the table of contents)
        if len(risk_iter) > 1:
            upper_text_risk = upper_text[risk_iter[1].start():]

        # Extract text between Item 1A and the next item header
        risks_match = re.search(
            r"(?is)ITEM\s*1A\.?\s*RISK FACTORS(.*?)(ITEM\s*1B\.|ITEM\s*2\.|ITEM\s*3\.|ITEM\s*4\.)",
            upper_text_risk
        )

        # If match found, extract group(1); else empty
        risks_text = risks_match.group(1).strip() if risks_match else ""

        # ------------------------------
        # Extract MD&A section
        # ------------------------------

        upper_text_mda = None

        # Look for Item 7 MD&A or Item 2 (for small reporting companies)
        mda_iter = list(re.finditer(
            r"ITEM\s*(7|2)\.?\s*MANAGEMENT[^A-Z]{0,5}?S\s+DISCUSSION\s+AND\s+ANALYSIS",
            upper_text
        ))

        # If second occurrence exists, skip TOC and use it
        if len(mda_iter) > 1:
            upper_text_mda = upper_text[mda_iter[1].start():]

        # If only one occurrence, use that
        elif len(mda_iter) == 1:
            upper_text_mda = upper_text[mda_iter[0].start():]

        # If nothing found, fall back on full text
        else:
            upper_text_mda = upper_text

        # Extract MD&A text between Item 7/2 and the next section
        mda_match = re.search(
            r"(?is)ITEM\s*(7|2)\.?\s*MANAGEMENT[^A-Z]{0,5}?S\s+DISCUSSION\s+AND\s+ANALYSIS"
            r"(.*?)(?=ITEM\s*(7A|3)\.|ITEM\s*8\.|PART\s*III\.)",
            upper_text_mda
        )

        # Extract group(2) containing the actual MD&A narrative
        mda_text = mda_match.group(2).strip() if mda_match else ""

        # Return: full text (cleaned), risk section, MD&A section
        return (cleaned_text, risks_text, mda_text)

    except Exception as e:
        # Print error and fail gracefully
        print(f"Error parsing {filing_url}: {e}")
        return ("", "", "")


### Create and save a new table with the extracted text columns

##### Parse the top 100 data filings into the functions and create new text columns

In [0]:
# Import PySpark UDF tools and functions
from pyspark.sql.functions import udf
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

# ----------------------------------------------------------
# Create a UDF that wraps the 'parsing' function
# The UDF returns a struct with 3 string fields:
#   - cleaned_text
#   - risks_text
#   - mda_text
# ----------------------------------------------------------
parse_udf = udf(
    parsing,
    StructType([
        StructField("cleaned_text", StringType()),
        StructField("risks_text", StringType()),
        StructField("mda_text", StringType())
    ])
)

# ----------------------------------------------------------
# Apply the UDF to each filing_url row in top100_raw_files
# This creates a new column "parsed" that is a struct
# containing the output of parsing():
#   parsed.cleaned_text
#   parsed.risks_text
#   parsed.mda_text
# ----------------------------------------------------------
parsed_df = (
    top100_raw_files
        # Apply UDF to filing_url
        .withColumn("parsed", parse_udf(F.col("filing_url")))

        # Select original metadata + extracted NLP text
        .select(
            "cik",
            "company_name",
            "tickers",
            "form_type",
            "filing_date",
            "accessionNumber",
            "market_cap",
            "sector",
            "filing_url",

            # Extract fields inside struct "parsed"
            F.col("parsed.cleaned_text").alias("text"),
            F.col("parsed.risks_text").alias("risks_text"),
            F.col("parsed.mda_text").alias("mda_text")
        )
)

# Show results in Databricks notebook UI
display(parsed_df)


In [0]:
display(parsed_df)

##### Save the new dataframe in a table

In [0]:
# save the extracted data in a new table
parsed_df.write.format("delta").option("mergeSchema", "true").mode("overwrite").saveAsTable("sec_filings.top100_clean_filing")

In [0]:
spark.sql("USE CATALOG workspace")
# spark.sql("SHOW SCHEMAS IN workspace").display()
spark.sql("SHOW TABLES IN sec_filings").display()

In [0]:
spark.sql("USE sec_filings;")
spark.sql("DROP TABLE IF EXISTS market_trend1;")
spark.sql("DROP TABLE IF EXISTS market_trend2;")
spark.sql("DROP TABLE IF EXISTS market_trend3;")
spark.sql("DROP TABLE IF EXISTS market_trend4;")
spark.sql("DROP TABLE IF EXISTS market_trend5;")
spark.sql("DROP TABLE IF EXISTS market_trend6;")